---

## **DIPLOME UNIVERSITAIRE SDA**

## **ML Ops**

---

---
## **Prédiction de défaut de paiement (Loan Default)**

## **NBx_[STEP] : [TITRE DU NOTEBOOK]**

> Nommage du fichier : `NBx_[step].ipynb` (ex: NB1_EDA.ipynb, NB2_PREPROCESSING.ipynb, NB3_MODELISATION.ipynb)
---

### **Contexte**

[Description courte de l'objectif de ce notebook et de sa place dans le pipeline]

### **Données**

- **Dataset** : `Loan_Data.csv` - 10 000 clients bancaires (synthétique)
- **Variable cible** : `default` (binaire : 0 = pas de défaut, 1 = défaut)
- **Modèles imposés** : Decision Tree, Régression Logistique, Random Forest

---

---

### Plan du notebook

| Section | Contenu |
|---------|--------|
| 1. Configuration | Imports, chemins relatifs, seed, versions |
| 2. [SECTION] | [CONTENU] |
| 3. [SECTION] | [CONTENU] |
| 4. Conclusion | [CONTENU] |

---

---

### Objectif du notebook

Ce notebook est un **livrable de pipeline**. Il couvre [DÉCRIRE LE PÉRIMÈTRE].

Il est conçu pour être :
- **reproductible** (chemins relatifs, seed fixe),
- **idempotent** (relançable sans effet de bord),
- **traçable** (quality gates go/no-go explicites, assertions),
- **orienté décisions** : chaque sortie justifie un choix pour le notebook suivant.

---

In [ ]:
# 1.1. Imports
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ajouter ici les imports specifiques au notebook
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler

print(">> 1.1. Imports : OK")

In [ ]:
# 1.2. Chemins relatifs (pipeline)
BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = BASE / "model"
MODEL_DIR.mkdir(exist_ok=True)

CSV_PATH = DATA_DIR / "Loan_Data.csv"
assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH}"

# Convention de nommage des figures :
# OUTPUT_DIR / "NBx_nom_figure.png" (ex: NB1_eda_boxplots.png, NB2_feature_importance.png)
#
# Convention de sauvegarde des graphiques :
# plt.tight_layout()                                        # ajuster les marges
# plt.savefig(OUTPUT_DIR / "NBx_nom_figure.png", dpi=150)   # sauvegarder (dpi=150 standard projet)
# plt.show()                                                 # afficher dans le notebook

print(f"Base    : {BASE}")
print(f"Fichier : {CSV_PATH}")
print(">> 1.2. Chemins : OK")

In [ ]:
# 1.3. Versions / seed
print("python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("seaborn :", sns.__version__)

SEED = 42
np.random.seed(SEED)

# Toujours utiliser random_state=SEED dans sklearn
# Exemple : train_test_split(..., random_state=SEED)
# Exemple : DecisionTreeClassifier(random_state=SEED)

print(f"SEED    : {SEED}")
print(">> 1.3. Versions / seed : OK")

In [ ]:
# 1.4. Constantes du projet (heritées du NB1_EDA — ne pas modifier)

CIBLE = "default"
COLONNE_A_SUPPRIMER = ["customer_id"]
FEATURES = [
    "credit_lines_outstanding",
    "loan_amt_outstanding",
    "total_debt_outstanding",
    "income",
    "years_employed",
    "fico_score",
]
MODELES_IMPOSES = ["Decision Tree", "Regression Logistique", "Random Forest"]
TEST_SIZE = 0.2
STRATIFY = True

# Stratégie d'évaluation (métriques)
METRIC_PRIMAIRE = "f1"
METRICS_SECONDAIRES = ["recall", "precision", "roc_auc", "average_precision"]

print(">> 1.4. Constantes projet : OK")

In [ ]:
# 2.1. Chargement des données
df = pd.read_csv(CSV_PATH)
assert len(df) == 10_000, f"Attendu 10 000 lignes, obtenu {len(df)}"
print(f"Dataset charge : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

In [ ]:
# 2.1b. Quality gates (chaque check définit explicitement sa condition de succès)
checks = {
    "nb_lignes": (len(df), len(df) == 10_000),
    "nb_colonnes": (len(df.columns), len(df.columns) == 8),
    "doublons": (df.duplicated().sum(), df.duplicated().sum() == 0),
    "valeurs_manquantes_total": (df.isnull().sum().sum(), df.isnull().sum().sum() == 0),
    "customer_id_unique": (df["customer_id"].nunique(), df["customer_id"].nunique() == len(df)),
}

all_ok = True
for k, (valeur, condition) in checks.items():
    status = "[OK]" if condition else "[KO]"
    if not condition:
        all_ok = False
    print(f"  {status} {k}: {valeur}")

assert all_ok, "Quality gates KO — vérifier les données avant de continuer"
print(">> 2.1b. Quality gates : OK")

---

### Patterns de modélisation disponibles

Les cellules ci-dessous sont des **exemples à adapter**. Supprimer celles qui ne sont pas utiles au notebook.

---

In [ ]:
# Pattern : diagnostic NaN et infinis après création de features
# À utiliser après toute transformation qui peut générer des NaN ou Inf (division, log, etc.)
colonnes_a_verifier = ["colonne_1", "colonne_2"]  # adapter

nan_count = df[colonnes_a_verifier].isnull().sum()
inf_count = np.isinf(df[colonnes_a_verifier]).sum()

print("=== Diagnostic post-feature engineering ===")
for col in colonnes_a_verifier:
    print(f"  {col} : {nan_count[col]} NaN, {inf_count[col]} Inf")

# Traitement : remplacer les infinis par NaN, puis imputer par la médiane
df.replace([np.inf, -np.inf], np.nan, inplace=True)
for col in colonnes_a_verifier:
    mediane = df[col].median()
    nb_impute = df[col].isnull().sum()
    if nb_impute > 0:
        df[col].fillna(mediane, inplace=True)
        print(f"  {col} : {nb_impute} valeurs imputées par la médiane ({mediane:.4f})")

assert df[colonnes_a_verifier].isnull().sum().sum() == 0, "NaN restants après imputation"
assert np.isinf(df[colonnes_a_verifier]).sum().sum() == 0, "Infinis restants après imputation"
print(">> Diagnostic NaN/Inf : OK")

In [ ]:
# Pattern : audit d'intégrité du dataset avant preprocessing
# À utiliser avant tout split ou entraînement pour vérifier la propreté des données
features_a_auditer = FEATURES  # adapter selon le notebook

X_audit = df[features_a_auditer]
audit = {
    "shape": (X_audit.shape, True),
    "doublons": (X_audit.duplicated().sum(), X_audit.duplicated().sum() >= 0),
    "infinis": (np.isinf(X_audit).sum().sum(), np.isinf(X_audit).sum().sum() == 0),
    "NaN_total": (X_audit.isnull().sum().sum(), X_audit.isnull().sum().sum() == 0),
    "features_mortes_std0": ((X_audit.std() == 0).sum(), (X_audit.std() == 0).sum() == 0),
}

print("=== Audit d'intégrité ===")
all_ok = True
for k, (valeur, condition) in audit.items():
    status = "[OK]" if condition else "[KO]"
    if not condition:
        all_ok = False
    print(f"  {status} {k}: {valeur}")

assert all_ok, "Audit KO — corriger avant de continuer"
print(">> Audit intégrité : OK")

In [ ]:
# 2.2. Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Suppression de la colonne non informative
df = df.drop(columns=COLONNE_A_SUPPRIMER)

# Séparation features / cible
X = df[FEATURES]
y = df[CIBLE]

# Split train / test (stratifié pour conserver le ratio de la cible)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

# Vérification du ratio (quality gate)
print(f"Train : {X_train.shape[0]} lignes | ratio défaut : {y_train.mean():.3f}")
print(f"Test  : {X_test.shape[0]} lignes | ratio défaut : {y_test.mean():.3f}")

# Cross-validation : 5 folds stratifiés
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(">> 2.2. Preprocessing : OK")

---

### Patterns de validation disponibles

Les cellules ci-dessous sont des **exemples à adapter**. Supprimer celles qui ne sont pas utiles au notebook.

---

In [ ]:
# Pattern : entraînement avec Pipeline + chrono
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score

# Pipeline LR : scaler + modèle (le scaler est dans le pipeline pour éviter le data leakage)
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
])

# Pipeline DT : pas besoin de scaler (invariant aux échelles)
pipe_dt = Pipeline([
    ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=SEED)),
])

# Pipeline RF : pas besoin de scaler
pipe_rf = Pipeline([
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=SEED)),
])

# Entraînement + mesure du temps
resultats = []
for nom, pipe in [("Regression Logistique", pipe_lr), ("Decision Tree", pipe_dt), ("Random Forest", pipe_rf)]:
    t0 = time.time()
    pipe.fit(X_train, y_train)
    duree = round(time.time() - t0, 2)

    y_pred = pipe.predict(X_test)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")

    resultats.append({
        "modele": nom,
        "f1_test": round(f1_score(y_test, y_pred), 4),
        "recall_test": round(recall_score(y_test, y_pred), 4),
        "precision_test": round(precision_score(y_test, y_pred), 4),
        "accuracy_test": round(accuracy_score(y_test, y_pred), 4),
        "f1_cv_mean": round(cv_scores.mean(), 4),
        "f1_cv_std": round(cv_scores.std(), 4),
        "duree_s": duree,
        "seed": SEED,
    })
    print(f"{nom} : F1={resultats[-1]['f1_test']} | CV={resultats[-1]['f1_cv_mean']}+-{resultats[-1]['f1_cv_std']} | {duree}s")

In [ ]:
# Pattern : tableau comparatif des résultats
df_resultats = pd.DataFrame(resultats)
print("=" * 60)
print("COMPARAISON DES MODÈLES")
print("=" * 60)
print(df_resultats.to_string(index=False))

# Sauvegarde des résultats en CSV
df_resultats.to_csv(OUTPUT_DIR / "NBx_comparaison_modeles.csv", index=False)
print(f"\nRésultats sauvegardés dans {OUTPUT_DIR / 'NBx_comparaison_modeles.csv'}")

---
## Conclusion

### Ce qui a été fait dans ce notebook

| Constat | Preuve | Décision pour le notebook suivant |
|---------|--------|------------------------------------|
| [Constat 1] | [Preuve / métrique] | [Décision] |
| [Constat 2] | [Preuve / métrique] | [Décision] |

### Limites

- [Limite 1]
- [Limite 2]

### Choix retenus pour le notebook suivant

- [Choix 1]
- [Choix 2]

---